# Coste de los límites semánticos

Experimento de `uibcdf/recorda-lab#2`. Compara la misma función directa, decorada sin
sesión, con sesión y omisiones, y con sesión y referencias nativas. Se conserva el
`fsync` real del prototipo. No hay umbrales de tiempo en los tests.

Los tiempos de una función muy barata pueden fluctuar e incluso producir diferencias
negativas por ruido. Mira las observaciones y sus rangos, además de las medianas.
Estos datos ayudan a elegir límites semánticos: no prueban utilidad científica real.

In [ ]:
import json
import sys
from pathlib import Path
from uuid import uuid4

lab_root = Path.cwd()
if lab_root.name == "notebooks":
    lab_root = lab_root.parent
assert (lab_root / "experiments").is_dir(), "Abre el notebook desde recorda-lab"
sys.path.insert(0, str(lab_root / "experiments"))
run_dir = lab_root / "artifacts" / f"notebook-{uuid4().hex}"
run_dir.mkdir(parents=True, exist_ok=False)
print(sys.executable)
print(run_dir)

In [ ]:
from run_performance import run

measurements = run(
    run_dir / "performance",
    repetitions=5,
    baseline_iterations=1000,
    active_iterations=20,
    size=10000,
)

Cada operación activa escribe tres eventos y sincroniza cada uno. El tiempo medido
incluye la captura, la ejecución, la persistencia y, cuando corresponde, el cálculo de
referencias. Se excluyen la apertura/cierre de la sesión y la inspección de resultados.

In [ ]:
from IPython.display import Markdown, display

lines = [
    "| Trabajo | Directa (µs) | Inactiva (µs) | Activa: omisiones (µs) | Activa: referencias (µs) |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for name, summary in measurements["summaries"].items():
    modes = summary["modes"]
    values = [
        modes[mode]["median_wall_ns_per_call"] / 1000
        for mode in ("direct", "inactive", "active_omissions", "active_references")
    ]
    lines.append("| " + name + " | " + " | ".join(f"{value:.3f}" for value in values) + " |")
display(Markdown("\n".join(lines)))

In [ ]:
observations = measurements["observations"]
print("Observaciones:", len(observations))
print("Artefacto:", run_dir / "performance" / "measurements.json")
assert all(
    row["events"] == 3 * row["iterations"] + 2
    for row in observations
    if row["mode"].startswith("active_")
)
assert json.loads((run_dir / "performance" / "measurements.json").read_text()) == measurements

Para la medición completa y repetible desde terminal:
`python experiments/run_performance.py artifacts/performance-check`.

Los journals y los resultados verificados acompañan a las observaciones. El coste
de generar una referencia puede crecer con el tamaño de los datos; en una integración
real conviene reutilizar identidades nativas ya disponibles cuando corresponda.
No se ha añadido una política de buffering ni se ha cambiado la durabilidad del prototipo.